# CNN Fear & Greed Index Data Updater

This notebook:
1. Loads the most recent `all_fng_csv_*.csv` file
2. Fetches latest data from CNN API
3. Concatenates new data points
4. Saves as `all_fng_csv_{day}_{month}_{year}.csv`

**API Endpoint:** https://production.dataviz.cnn.io/index/fearandgreed/graphdata/

In [14]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
import glob

In [15]:
# Configuration
BASE_URL = "https://production.dataviz.cnn.io/index/fearandgreed/graphdata/"
DATA_DIR = Path('data')
CURRENT_DATE = datetime.now().strftime('%Y-%m-%d')
# Format: day_month_year (e.g., 13_feb_2026)
OUTPUT_SUFFIX = datetime.now().strftime('%d_%b_%Y').lower()

print(f"Current date: {CURRENT_DATE}")
print(f"Output suffix: {OUTPUT_SUFFIX}")

Current date: 2026-02-13
Output suffix: 13_feb_2026


## Step 1: Find and Load Most Recent FNG File

In [16]:
# Find all existing FNG CSV files
fng_files = glob.glob(str(DATA_DIR / 'all_fng_csv_*.csv'))

if not fng_files:
    # Fallback to root directory
    fng_files = glob.glob('all_fng_csv*.csv')

if not fng_files:
    print("ERROR: No existing FNG CSV files found!")
    print("Expected pattern: all_fng_csv_*.csv or all_fng_csv.csv")
    raise FileNotFoundError("No FNG data file found")

# Sort by modification time and get the most recent
most_recent_file = max(fng_files, key=lambda x: Path(x).stat().st_mtime)

print(f"Found {len(fng_files)} FNG file(s)")
print(f"\nMost recent file: {most_recent_file}")
print(f"Last modified: {datetime.fromtimestamp(Path(most_recent_file).stat().st_mtime)}")

Found 1 FNG file(s)

Most recent file: data/all_fng_csv_4_dec_2025.csv
Last modified: 2025-12-04 16:52:04.242279


In [17]:
# Load the most recent FNG data
fng_existing = pd.read_csv(most_recent_file)

# Standardize column names to lowercase for consistent processing
if 'Date' in fng_existing.columns:
    fng_existing = fng_existing.rename(columns={'Date': 'date', 'Fear Greed': 'fng'})
elif 'date' not in fng_existing.columns:
    # Assume first column is date, second is FNG value
    fng_existing.columns = ['date', 'fng']

fng_existing['date'] = pd.to_datetime(fng_existing['date'])

print(f"Loaded existing data: {len(fng_existing)} records")
print(f"Date range: {fng_existing['date'].min().date()} to {fng_existing['date'].max().date()}")
print(f"\nLast 5 records:")
print(fng_existing.tail())

Loaded existing data: 5450 records
Date range: 2011-01-03 to 2025-12-04

Last 5 records:
           date   fng
5445 2025-11-30   0.0
5446 2025-12-01  24.0
5447 2025-12-02  23.0
5448 2025-12-03  36.0
5449 2025-12-04  38.0


## Step 2: Fetch Latest Data from CNN API

In [18]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
}

print("Fetching data from CNN API...")
print(f"URL: {BASE_URL}")

response = requests.get(BASE_URL, headers=headers, timeout=30)

print(f"Status code: {response.status_code}")

if response.status_code != 200:
    print(f"ERROR: API returned status {response.status_code}")
    print(f"Response: {response.text}")
    raise Exception(f"API request failed with status {response.status_code}")

print("✓ Successfully fetched data from API")

Fetching data from CNN API...
URL: https://production.dataviz.cnn.io/index/fearandgreed/graphdata/
Status code: 200
✓ Successfully fetched data from API


In [19]:
# Parse JSON response
api_data = response.json()

print("\nAPI Response structure:")
print(f"  Top-level keys: {list(api_data.keys())}")

if 'fear_and_greed_historical' in api_data:
    historical_data = api_data['fear_and_greed_historical']['data']
    print(f"  Historical data points: {len(historical_data)}")
    print(f"\n  Sample data point:")
    print(f"    {historical_data[0]}")
else:
    print("ERROR: No 'fear_and_greed_historical' key in response!")
    print(f"Available keys: {list(api_data.keys())}")
    raise KeyError("Expected 'fear_and_greed_historical' not found in API response")


API Response structure:
  Top-level keys: ['fear_and_greed', 'fear_and_greed_historical', 'market_momentum_sp500', 'market_momentum_sp125', 'stock_price_strength', 'stock_price_breadth', 'put_call_options', 'market_volatility_vix', 'market_volatility_vix_50', 'junk_bond_demand', 'safe_haven_demand']
  Historical data points: 254

  Sample data point:
    {'x': 1739491200000.0, 'y': 43.68571428571429, 'rating': 'fear'}


## Step 3: Convert API Data to DataFrame

In [20]:
# Convert API data to DataFrame
api_records = []

for point in historical_data:
    timestamp_ms = int(point['x'])
    date = datetime.fromtimestamp(timestamp_ms / 1000)
    # Round to nearest integer using standard rounding (0.5 rounds up)
    value = int(float(point['y']) + 0.5)
    
    api_records.append({
        'date': date,
        'fng': value
    })

fng_api = pd.DataFrame(api_records)
fng_api['date'] = pd.to_datetime(fng_api['date']).dt.normalize()  # Remove time component

print(f"Converted {len(fng_api)} API records to DataFrame")
print(f"Date range: {fng_api['date'].min().date()} to {fng_api['date'].max().date()}")
print(f"\n📅 Earliest date available from API: {fng_api['date'].min().date()}")
print(f"📅 Latest date available from API: {fng_api['date'].max().date()}")
print(f"\nFirst 5 records from API:")
print(fng_api.head())

Converted 254 API records to DataFrame
Date range: 2025-02-13 to 2026-02-13

📅 Earliest date available from API: 2025-02-13
📅 Latest date available from API: 2026-02-13

First 5 records from API:
        date  fng
0 2025-02-13   44
1 2025-02-17   47
2 2025-02-18   48
3 2025-02-19   44
4 2025-02-20   37


## Step 4: Identify New and Overlapping Records

In [21]:
# Find the latest date in existing data
latest_existing_date = fng_existing['date'].max()
print(f"Latest date in existing data: {latest_existing_date.date()}")

# Find new records (dates after the latest existing date)
fng_new = fng_api[fng_api['date'] > latest_existing_date].copy()
print(f"New records from API: {len(fng_new)}")

if len(fng_new) > 0:
    print(f"\nNew date range: {fng_new['date'].min().date()} to {fng_new['date'].max().date()}")
    print(f"\nNew records preview:")
    print(fng_new.head(10))
else:
    print("\nNo new records to add (data is up to date)")

# Also update any overlapping dates (in case API values changed)
fng_overlap = fng_api[fng_api['date'] <= latest_existing_date].copy()
print(f"\nOverlapping records to update: {len(fng_overlap)}")

Latest date in existing data: 2025-12-04
New records from API: 50

New date range: 2025-12-07 to 2026-02-13

New records preview:
          date  fng
204 2025-12-07   31
205 2025-12-08   32
206 2025-12-09   40
207 2025-12-10   48
208 2025-12-11   42
209 2025-12-14   52
210 2025-12-15   45
211 2025-12-16   47
212 2025-12-17   44
213 2025-12-18   50

Overlapping records to update: 204


## Step 5: Merge and Concatenate Data

In [22]:
# Merge: Update overlapping dates, then append new dates
fng_combined = fng_existing.copy()

# Update overlapping dates
if len(fng_overlap) > 0:
    # Create a set of dates to update for faster lookup
    overlap_dates = set(fng_overlap['date'])
    
    # Remove old values for overlapping dates
    fng_combined = fng_combined[~fng_combined['date'].isin(overlap_dates)]
    
    # Add updated values from API
    fng_combined = pd.concat([fng_combined, fng_overlap], ignore_index=True)
    print(f"Updated {len(fng_overlap)} overlapping records")

# Append new records
if len(fng_new) > 0:
    fng_combined = pd.concat([fng_combined, fng_new], ignore_index=True)
    print(f"Added {len(fng_new)} new records")

# Sort by date and reset index
fng_combined = fng_combined.sort_values('date').reset_index(drop=True)

print(f"\nCombined dataset: {len(fng_combined)} records")
print(f"Date range: {fng_combined['date'].min().date()} to {fng_combined['date'].max().date()}")

Updated 204 overlapping records
Added 50 new records

Combined dataset: 5500 records
Date range: 2011-01-03 to 2026-02-13


## Step 5.5: Replace 2020-2021 & 2022 Jan-Jun with Scraped Data

The scraped data is more accurate for this period due to date shift corrections.
- Replaces: 2020 (full year), 2021 (full year), 2022 (Jan-Jun only)
- Keeps: 2022 (Jul-Dec) from CNN API

In [23]:
# Find most recent scraped file
scraped_files = glob.glob(str(DATA_DIR / 'fng_scraped_*.csv'))

if scraped_files:
    most_recent_scraped = max(scraped_files, key=lambda x: Path(x).stat().st_mtime)
    print(f"Found scraped file: {Path(most_recent_scraped).name}")
    
    # Load scraped data
    df_scraped = pd.read_csv(most_recent_scraped)
    df_scraped['date'] = pd.to_datetime(df_scraped['date'])
    
    # Filter 2020, 2021, and 2022 (Jan-June only) from scraped data
    scraped_replace = df_scraped[
        (df_scraped['date'].dt.year == 2020) |
        (df_scraped['date'].dt.year == 2021) |
        ((df_scraped['date'].dt.year == 2022) & (df_scraped['date'].dt.month <= 6))
    ].copy()
    
    if len(scraped_replace) > 0:
        print(f"  Found {len(scraped_replace)} scraped records for 2020-2021 and 2022 Jan-Jun")
        
        # Remove 2020, 2021, and 2022 Jan-Jun from combined data
        original_count = len(fng_combined)
        fng_combined = fng_combined[
            ~((fng_combined['date'].dt.year == 2020) |
              (fng_combined['date'].dt.year == 2021) |
              ((fng_combined['date'].dt.year == 2022) & (fng_combined['date'].dt.month <= 6)))
        ]
        removed_count = original_count - len(fng_combined)
        print(f"  Removed {removed_count} original records")
        
        # Add scraped data
        scraped_replace_renamed = scraped_replace.rename(columns={'date': 'date', 'value': 'fng'})
        fng_combined = pd.concat([fng_combined, scraped_replace_renamed[['date', 'fng']]], ignore_index=True)
        fng_combined = fng_combined.sort_values('date').reset_index(drop=True)
        
        print(f"  ✓ Replaced 2020-2021 and 2022 Jan-Jun with scraped data")
        print(f"  New total: {len(fng_combined)} records")
    else:
        print("  No data found in scraped file")
else:
    print("⚠️  No scraped file found - skipping replacement")
    print("  (This is optional - continuing without replacement)")

Found scraped file: fng_scraped_13_feb_2026.csv
  Found 629 scraped records for 2020-2021 and 2022 Jan-Jun
  Removed 912 original records
  ✓ Replaced 2020-2021 and 2022 Jan-Jun with scraped data
  New total: 5217 records


## Step 6: Fill Missing Dates

- Weekends (Saturday & Sunday) are set to **0**
- Weekday holidays are **forward-filled** with the last known value

In [24]:
# Create full date range
date_range = pd.date_range(
    start=fng_combined['date'].min(),
    end=fng_combined['date'].max(),
    freq='D'
)

# Find missing dates
existing_dates = set(fng_combined['date'])
missing_dates = [d for d in date_range if d not in existing_dates]

print(f"Total date range: {len(date_range)} days")
print(f"Missing dates: {len(missing_dates)}")

if len(missing_dates) > 0:
    # Add missing dates with NaN values
    missing_df = pd.DataFrame({
        'date': missing_dates,
        'fng': np.nan
    })
    
    fng_combined = pd.concat([fng_combined, missing_df], ignore_index=True)
    fng_combined = fng_combined.sort_values('date').reset_index(drop=True)
    print(f"Added {len(missing_dates)} missing dates")

# Set ALL Saturdays and Sundays to 0 (regardless of existing values)
fng_combined['dayofweek'] = fng_combined['date'].dt.dayofweek
weekend_mask = fng_combined['dayofweek'].isin([5, 6])
fng_combined.loc[weekend_mask, 'fng'] = 0.0

weekend_count = weekend_mask.sum()
print(f"Set {weekend_count} weekend days (Sat/Sun) to 0")

# Forward fill remaining missing values (weekday holidays only)
remaining_na = fng_combined['fng'].isna().sum()
if remaining_na > 0:
    fng_combined['fng'] = fng_combined['fng'].ffill().bfill()
    print(f"Forward-filled {remaining_na} missing weekday values")

# Drop helper column
fng_combined = fng_combined.drop('dayofweek', axis=1)

print(f"\nFinal dataset: {len(fng_combined)} records")

Total date range: 5521 days
Missing dates: 304
Added 304 missing dates
Set 1576 weekend days (Sat/Sun) to 0
Forward-filled 33 missing weekday values

Final dataset: 5521 records


## Step 7: Save Updated Data

In [25]:
# Prepare output filename with day_month_year format
output_filename = f'all_fng_csv_{OUTPUT_SUFFIX}.csv'
output_path = DATA_DIR / output_filename

# Ensure data directory exists
DATA_DIR.mkdir(exist_ok=True)

# Save with standard column names and proper formatting
fng_output = fng_combined.copy()
# Ensure fng values are floats (will display as 68.0, 67.0, etc.)
fng_output['fng'] = fng_output['fng'].astype(float)
# Rename to match existing format
fng_output.columns = ['Date', 'Fear Greed']

fng_output.to_csv(output_path, index=False)

print(f"✓ Saved updated data to: {output_path}")
print(f"\nFinal dataset summary:")
print(f"  Total records: {len(fng_output):,}")
print(f"  Date range: {fng_output['Date'].min()} to {fng_output['Date'].max()}")
print(f"  Non-null values: {fng_output['Fear Greed'].notna().sum():,}")
print(f"  Missing values: {fng_output['Fear Greed'].isna().sum():,}")

print(f"\nLast 10 records:")
print(fng_output.tail(10))

✓ Saved updated data to: data/all_fng_csv_13_feb_2026.csv

Final dataset summary:
  Total records: 5,521
  Date range: 2011-01-03 00:00:00 to 2026-02-13 00:00:00
  Non-null values: 5,521
  Missing values: 0

Last 10 records:
           Date  Fear Greed
5511 2026-02-04        34.0
5512 2026-02-05        44.0
5513 2026-02-06        44.0
5514 2026-02-07         0.0
5515 2026-02-08         0.0
5516 2026-02-09        51.0
5517 2026-02-10        48.0
5518 2026-02-11        36.0
5519 2026-02-12        36.0
5520 2026-02-13        36.0


## Summary Statistics

In [26]:
print("="*80)
print("UPDATE SUMMARY")
print("="*80)
print(f"\nInput file:  {most_recent_file}")
print(f"  Records: {len(fng_existing):,}")
print(f"  Latest date: {fng_existing['date'].max().date()}")

print(f"\nAPI data:")
print(f"  Records fetched: {len(fng_api):,}")
print(f"  📅 Earliest date from API: {fng_api['date'].min().date()}")
print(f"  📅 Latest date from API: {fng_api['date'].max().date()}")
print(f"  Coverage: {(fng_api['date'].max() - fng_api['date'].min()).days} days")

print(f"\nOutput file: {output_path}")
print(f"  Records: {len(fng_output):,}")
print(f"  Latest date: {pd.to_datetime(fng_output['Date']).max().date()}")
print(f"  New records added: {len(fng_new):,}")
print(f"  Records updated: {len(fng_overlap):,}")

print(f"\n✓ Update complete!")

UPDATE SUMMARY

Input file:  data/all_fng_csv_4_dec_2025.csv
  Records: 5,450
  Latest date: 2025-12-04

API data:
  Records fetched: 254
  📅 Earliest date from API: 2025-02-13
  📅 Latest date from API: 2026-02-13
  Coverage: 365 days

Output file: data/all_fng_csv_13_feb_2026.csv
  Records: 5,521
  Latest date: 2026-02-13
  New records added: 50
  Records updated: 204

✓ Update complete!
